---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [61]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018" # Which claims year to load # TODO: maybe add a script that loops through all claims?
split_parts <- 15 # How many (integer) parts to split the 12+m row claims file into # TODO: a value of 10 for claims year 2018 leads to quoted newline errors
end_nrow <- Inf # How many rows/entries to show in summary tables
gcp_proj <- if (.Platform$OS.type == "unix") system("gcloud config get-value project", intern = TRUE) else NULL
max_rows <- 15000 # Max rows to return for bq query
cat(paste("GCP Project:", gcp_proj, "\n"))
ver_to_use <- "latest" # Must be set to latest, local and bak have been deleted
encode <- "unknown" # Choices: unknown, UTF-8, Latin-1
sep <- "," # Choices: "," or "\t"

# Input:
to_read <- FALSE # TODO: Deprecated, used to be whether to forcibly read the whole file again instead of using the split parts created even if available
to_split <- TRUE # TODO: Deprecated, only used when to_sample is TRUE # Whether to split into split_parts parts (i.e. to fit in 32gb RAM).
to_sample <- TRUE # Whether to sample each split_parts part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 125 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor. Choose between 5, 25, and 125

# Output:
to_write <- TRUE # Whether to write out intermediate files and caches (i.e. part files, sample files). TODO: upload to BQ as well
to_group <- TRUE # Whether to export for the batch grouper or not

# Debug:
to_debug <- FALSE # whether to print debug statements
to_profvis <- FALSE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallel <- FALSE # Whether to view intermediate per part/chunk checks and print statements (not consolidated) when parallelized
to_parallel <- TRUE # Whether to parallelize each split_parts part into availableCores() - 1 chunks. Cuts down processing time from 120min to 15min.
to_split_read <- FALSE # WARNING: TRUE uses a lot of memory!!
tmp_nrow <- Inf # Per part/chunk end_nrow (leave at Inf)
diff_chars <- 0

drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

seed <- 123 # Seed for reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)
set.seed(seed) # Setting the seed
global_seed <- seed # global_seed for future_lapply parts for parallelized operations

ram_size <- 32 # Input virtual or physical machine's RAM size here
ram_buffer <- 0.1 # How much of a buffer to leave for the OS
ram_limit <- (1 - ram_buffer) * (ram_size) * (1024^3) # Compute ram_limit in bytes
# Allowing each future_lapply session to use more memory
options(future.globals.maxSize = ram_limit)

# Compute the RAM limit for R processes, leaving the buffer for the OS
ram_limit_gb <- round((1 - ram_buffer) * ram_size, 0)
# Set the environment variable R_FUTURE_MAX_RAM in GB
# Sys.setenv(R_FUTURE_MAX_RAM = paste0(ram_limit_gb, "G"))

# Print the set RAM limit
# cat("Setting R_FUTURE_MAX_RAM to:", ram_limit_gb, "GB\n")
cat(sprintf("Setting future.globals.maxSize to: %.1f GB", ram_limit / (1024^3)))

if (!split_parts == as.integer(split_parts) || split_parts <= 1) stop("ERROR: split_parts must be an integer greater than or equal to 2!")
if (ram_size <= 64 && split_parts <= 2) stop("Please set split_parts to at least 3 for 64 GB machines or it will likely crash")
if (ram_size <= 32 && split_parts <= 4) stop("Please set split_parts to at least 5 for 32 GB machines or it will likely crash")
if (ram_size <= 32 && to_split_read == TRUE) stop("Please set to_split_read to TRUE for 32 GB machines or it will likely crash")


GCP Project: test-drg-pipeline 
Setting future.globals.maxSize to: 28.8 GB

## Load Required Libraries & Initial Functions

In [62]:
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading
library(here) # Library here() so scripts can be loaded


In [63]:
scripts <- list( # List of scripts to source
  libraries = "00_libraries.R",
  formats = "01_data-formats.R",
  paths = "02_file-paths.R",
  general = "03_general-functions.R",
  clean = "04a_clean-data-functions.R",
  chunk = "04b_chunk-functions.R",
  part = "04c_part-functions.R",
  io = "05_io-functions.R",
  icd = "06_icd-functions.R",
  rvs = "07_rvs-functions.R",
  pdx = "08_pdx-functions.R",
  grouper = "09_grouper-functions.R",
  timing = "10_timing-functions.R",
  debug = "11_debug-functions.R",
  summary = "12_summary-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts", script))


Directory already exists: data-cleaning/data-claims/intermediate 
Directory already exists: data-cleaning/cache 
Directory already exists: data-cleaning/data-aux-files 
Directory already exists: data-cleaning/data-excel 
Directory already exists: data-cleaning/data-claims/cleaned 
Directory already exists: data-cleaning/data-grouper-output 
Directory already exists: data-cleaning/data-claims/chunked 
Directory already exists: data-cleaning/data-claims/raw/parts 
Directory already exists: data-cleaning/data-claims/raw/samples 
Directory already exists: data-cleaning/data-claims/raw 
Directory already exists: data-cleaning/profvis 
Total Rows via cached object: 11777674

In [64]:
# TODO: figure out a way to return to default outputs
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


## Load claims files from GCS

In [65]:
# Loop through the years 2018 to 2021
for (year in 2018:2021) {
  file_name <- paste0("claims_extract_CLAIMS_", year, "_", ver_to_use, ".csv")
  bq_name <- paste0("claims_extract_CLAIMS_", year, "_", ver_to_use, ".csv")

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "test-drg-pipeline") {
      system(paste0("cd .. && gsutil cp gs://test-phic-claims-raw/", bq_name, " ", raw_claims_path),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not test-drg-pipeline")
    }
  } else {
    cat(paste("File", file_name, "already exists in the target directory. Skipping download.\n"))
  }
}


File claims_extract_CLAIMS_2018_latest.csv already exists in the target directory. Skipping download.
File claims_extract_CLAIMS_2019_latest.csv already exists in the target directory. Skipping download.
File claims_extract_CLAIMS_2020_latest.csv already exists in the target directory. Skipping download.
File claims_extract_CLAIMS_2021_latest.csv already exists in the target directory. Skipping download.


## Load Mapping Data

In [66]:
# Read in all rvs codes and turn to character for further processing
# Run the gcloud bq query command to save the result as a CSV file
if (!file.exists(here(aux_path, "proc.csv"))) {
  system(
    paste0(
      "bq query --use_legacy_sql=false --format=csv --max_rows=", max_rows,
      " 'SELECT * FROM `", gcp_proj, ".grouper_v5.proc`' > ", here(aux_path, "proc.csv")
    ),
    intern = FALSE, ignore.stderr = FALSE
  )
} else {
  warning("proc.csv already exists, skipping bq query")
}

# Read the CSV file into an R data frame
proc <- fread(here(aux_path, "proc.csv"))[, CODE := as.character(CODE)]

# Read in icd9cm equivalents of rvs codes,
# then convert to character and also remove decimals, whilst keeping trailing zeroes
if (!file.exists(here(aux_path, "rvs_icd9cm.csv"))) {
  system(
    paste0(
      "bq query --use_legacy_sql=false --format=csv --max_rows=", max_rows,
      " 'SELECT * FROM `", gcp_proj, ".phic.acr_rvs_map`' > ", here(aux_path, "rvs_icd9cm.csv")
    ),
    intern = FALSE, ignore.stderr = FALSE
  )
} else {
  warning("rvs_icd9cm.csv already exists, skipping bq query")
}

rvs_icd9 <- fread(here(aux_path, "rvs_icd9cm.csv"), select = c("rvs", "icd9cm"))[, rvs := as.character(rvs)][, icd9cm := as.character(icd9cm * 100)]

# Merge with proc from above, to be able to classify by DRGUSE
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)], by.x = "icd9cm", by.y = "CODE", all.x = TRUE)

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

if (!file.exists(here(aux_path, "acr_rvs.csv"))) {
  system(
    paste0(
      "bq query --use_legacy_sql=false --format=csv --max_rows=", max_rows,
      " 'SELECT * FROM `", gcp_proj, ".phic.acr_procedure`' > ", here(aux_path, "acr_rvs.csv")
    ),
    intern = FALSE, ignore.stderr = FALSE
  )
} else {
  warning("acr_rvs.csv already exists, skipping bq query")
}

# Read in PHIC all case rates
acr_rvs <- fread(here(aux_path, "acr_rvs.csv"))

if (!file.exists(here(aux_path, "i10.csv"))) {
  system(
    paste0(
      "bq query --use_legacy_sql=false --format=csv --max_rows=", max_rows,
      " 'SELECT * FROM `", gcp_proj, ".grouper_v5.i10`' > ", here(aux_path, "i10.csv")
    ),
    intern = FALSE, ignore.stderr = FALSE
  )
} else {
  warning("i10.csv already exists, skipping bq query")
}

# Read in the thai icd10 library
tdrg_icd10 <- fread(here(aux_path, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])


Warning message in eval(expr, envir, enclos):
“proc.csv already exists, skipping bq query”
Warning message in eval(expr, envir, enclos):
“rvs_icd9cm.csv already exists, skipping bq query”
Warning message in eval(expr, envir, enclos):
“acr_rvs.csv already exists, skipping bq query”
Warning message in eval(expr, envir, enclos):
“i10.csv already exists, skipping bq query”


## Read, Process, Export Data (Looping through all parts)

In [67]:
all_parts_summaries <- list() # initialize list for summaries
if (to_debug) processing_times <- numeric(split_parts) # initialize list for ETA
dim_dt <- vector() # initialize vector for dt dimensions
ncores <- if (parallelly::availableCores() > 16) { # detect number of cores available for parallelization
  parallelly::availableCores() - 2
} else if (parallelly::availableCores() > 8) {
  parallelly::availableCores() - 1
} else {
  parallelly::availableCores() - 0
}
print(paste("Utilizing", ncores, "of", parallelly::availableCores(), "available threads"))

# Define a codeblock to avoid repeating it twice when to_profvis is TRUE and again if FALSE
# Makes it easier to maintain as well, since we only need to modify one section instead of two
unified_block <- function() {
  # Start main execution logic
  split_and_save_parts() # Read, split, and save partial files

  # Start the parallelization session or remain sequential
  # TODO: mclapply (unix-only) might be faster than future_lapply
  if (to_parallel) {
    strat <- if (.Platform$OS.type == "unix") multicore else multisession
    plan(strat, workers = ncores)
  }

  # For each partial file (part) of 1:N (split_parts) files,
  for (part in 1:split_parts) {
    # Process the partial file with or without parallelization
    result <- process_part(
      part, ncores, to_view_checks,
      global_seed, tmp_nrow, rvs_icd9, tdrg_icd10,
      acc_pdx, to_parallel, to_write, to_group, to_sample,
      diff_chars
    )

    # Save partial summaries to a list
    all_parts_summaries[[part]] <- result$combined_summary

    # Save partial processing time to a list
    if (to_debug) processing_times[part] <- result$processing_time

    # Print status update and ETA
    # - VS Code: Updates are shown after complete execution
    # - Positron: Updates are shown live
    # - JupyterLab: Updates are shown after complete execution
    if (to_debug) print_status_update(part, split_parts, processing_times)

    if (part == 1) dim_dt <<- dim(result$dt)

    rm(result)
    gc()
  }

  # Summaries are consolidated from 15 split_parts * 8 chunks = 120 sub outputs
  print_summary_tables( # Print final summaries
    combine_parts_summaries(all_parts_summaries, tmp_nrow),
    end_nrow
  )

  if (to_parallel) plan(sequential) # end parallelization
  # End main execution logic
  if (to_debug) {
    return(NULL)
  } # debug
}

# Call the main function with or without profvis
if (to_profvis) {
  saveWidget(profvis({
    unified_block()
  }), here(profvis_path))
} else {
  unified_block()
}


[1] "Utilizing 8 of 8 available threads"




Rename Success:
 TRUE 



Table: ICD Text Normalization for clin_c1

|old_code      |new_code | count| differing_chars|
|:-------------|:--------|-----:|---------------:|
|p0001         |P0001    |     1|       0.2000000|
|A09.9         |A099     |     1|       0.2000000|
|FP 001        |FP001    |     1|       0.1666667|
|A09, E86.1    |A09E861  |     2|       0.1250000|
|J18.99, Y9    |J1899Y9  |     1|       0.1250000|
|B05.2+ J17    |B052J17  |     1|       0.1250000|
|J18.99, Y95   |J1899Y95 |    99|       0.1111111|
|B05.2+ J17.1* |B052J171 |    59|       0.1111111|
|E11.2+ N08.3* |E112N083 |    11|       0.1111111|
|A09.9 E86.1   |A099E861 |     8|       0.1111111|
|E14.2+ N08.3* |E142N083 |     6|       0.1111111|
|E10.2+ N08.3* |E102N083 |     5|       0.1111111|
|A01.0+ J17.0* |A010J170 |     4|       0.1111111|
|B01.2+ J17.1* |B012J171 |     3|       0.1111111|
|A09.9, E86.1  |A099E861 |     2|       0.1111111|
|B65.9+ J17.3* |B659J173 |     2|       0.1111111|
|B25.0+ J17

ERROR: Error in cat(head(processed_icd10_map, end_nrow)): argument 1 (type 'list') cannot be handled by 'cat'


## Runtime Estimation

In [ ]:
print_time_estimates() # Print time estimates along with estimate for full claims file


Time spent (total)               : 36.791 sec elapsed
Time spent (t/row) for 94.2k rows: 0.39 msec
Time (est) (total) for 11.8m rows: 76.65 min


## Debugging

In [ ]:
# in case we want to run this cell independently:
source(here::here("data-cleaning/r_scripts", "11_debug-functions.R"))

# Consolidate all r_scripts scripts into everything.R; useful for debugging
concatenate_r_files(here::here("data-cleaning/r_scripts"), here::here("data-cleaning/everything/everything.R"))


In [ ]:
rm(list = ls())
gc()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1143884,61.1,2409680,128.7,2409680,128.7
Vcells,2562573,19.6,8388608,64.0,8388608,64.0


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb --output everything/drg-cleaning

